# Lesson 9: Running a Pretrained Language Model

In the earlier Weird AI lessons, you built many of the core pieces of a language model from scratch. In this lesson, you will use a pretrained model so you can focus on the complete text-generation pipeline.

The pipeline is:

```text
prompt text -> tokenizer -> token IDs -> pretrained model -> generated token IDs -> generated text
```

By the end of this notebook, you should be able to explain how a prompt becomes token IDs, how a pretrained model generates more tokens, and how those generated tokens become readable text again.

## Section 1: Setup

Run this notebook from the project root folder. If imports fail, make sure your virtual environment is active and the project is installed in editable mode:

```bash
python -m pip install -e .
python -m pip install -r requirements.txt
```

This notebook uses Hugging Face `transformers`. If it is not installed yet, install it with:

```bash
python -m pip install transformers
```

In [18]:
from pathlib import Path
import torch

from weird_ai.pretrained import (
    DEFAULT_MODEL_NAME,
    get_device,
    load_tokenizer,
    load_model,
    tokenize_prompt,
    decode_generated_tokens,
    generate_text,
    generate_with_reasoning,
)

from weird_ai.reasoning import build_reasoning_prompt

## Section 2: Check Your Hardware

A GPU is helpful for larger models, but this lesson should work on CPU with a very small demonstration model. The starter code should choose CUDA when available, Apple Silicon MPS when available, and CPU otherwise.

In [19]:
# TODO: Run this cell after completing get_device() in pretrained.py.

device = get_device()
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.13.0+cpu


## Section 3: Load a Tokenizer

A tokenizer converts human-readable text into token IDs. The model does not directly understand strings such as `Write a parody about networking`. It receives integers.

For this notebook, the default model is intentionally tiny so the first download is fast.

In [20]:
# TODO: Run this cell after completing load_tokenizer() in pretrained.py.

tokenizer = load_tokenizer(DEFAULT_MODEL_NAME)
print(f"Loaded tokenizer for: {DEFAULT_MODEL_NAME}")
print(f"Pad token: {tokenizer.pad_token!r}")
print(f"EOS token: {tokenizer.eos_token!r}")

Loaded tokenizer for: sshleifer/tiny-gpt2
Pad token: '<|endoftext|>'
EOS token: '<|endoftext|>'


## Section 4: Tokenization Round Trip

A round trip means:

```text
text -> token IDs -> text
```

This helps verify that the tokenizer can encode and decode text.

In [21]:
prompt = "Write a parody chorus about a student debugging Python at midnight."

token_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

decoded_text = tokenizer.decode(token_ids[0], skip_special_tokens=True)

print("Original text:")
print(prompt)
print("Token IDs:")
print(token_ids)
print("Decoded text:")
print(decoded_text)

Original text:
Write a parody chorus about a student debugging Python at midnight.
Token IDs:
tensor([[16594,   257, 29695, 26441,   546,   257,  3710, 28769, 11361,   379,
         15896,    13]])
Decoded text:
Write a parody chorus about a student debugging Python at midnight.


### Reflection Check

Answer in your assignment document:

Why does the model need token IDs instead of regular text?
    The model cannot process text itself, it can only work with numbers. The model decides the next token using math and it cannot do math on text

## Section 5: Inspect Individual Tokens

Many modern tokenizers use subword tokens. A word may be one token, or it may be split into multiple pieces.

In [22]:
# TODO: After token_ids contains real token IDs, print each token ID and the text it decodes to.

for token_id in token_ids[0]:
    token_id = token_id.item()
    piece = tokenizer.decode([token_id])
    print(f"{token_id:>6} -> {piece!r}")

 16594 -> 'Write'
   257 -> ' a'
 29695 -> ' parody'
 26441 -> ' chorus'
   546 -> ' about'
   257 -> ' a'
  3710 -> ' student'
 28769 -> ' debugging'
 11361 -> ' Python'
   379 -> ' at'
 15896 -> ' midnight'
    13 -> '.'


## Section 6: Load a Pretrained Model

A pretrained model already has learned statistical patterns from a large training corpus. In this lesson, you are not training the model. You are using it for inference.

This can take a moment the first time because the model weights need to download.

In [23]:
# TODO: Run this cell after completing load_model() in pretrained.py.

model = load_model(DEFAULT_MODEL_NAME, device=device)
print(f"Loaded model for: {DEFAULT_MODEL_NAME}")

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model for: sshleifer/tiny-gpt2


## Section 7: Generate Text

Now you will complete the full pipeline:

```text
prompt -> tokenizer -> model.generate() -> decode
```

The `generate_text()` function in `pretrained.py` should handle those steps.

In [24]:
# TODO: Run this cell after completing generate_text() in pretrained.py.

prompt = "Write a short parody chorus about forgetting to commit code before a demo:"

output = generate_text(
    prompt,
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50,
    temperature=0.8,
    top_k=50,
    device=device,
)

print(output)

Write a short parody chorus about forgetting to commit code before a demo: Hancock conservationatisf Habit reviewing conservation reviewingpress Observ Hancock reborn hauleddit directly Prob stairsreement RhRocket heir hauledRocketreement confir credibilitySheroho reborn Danielimura Motorola directlyRocket Habit dispatchtingting heir directly Observ Participationpressting Money dispatch conservation Jr004 Danielpress


## Section 8: Compare Temperature Settings

Temperature changes how random generation is.

Lower temperature usually produces safer, more predictable text. Higher temperature usually produces more varied, surprising, and sometimes less coherent text.

In [11]:
prompt = "Write one funny line about a Wi-Fi router with stage fright:"

temperatures = [0.3, 0.8, 1.2]

for temp in temperatures:
    print("=" * 70)
    print(f"Temperature: {temp}")
    print("=" * 70)
    print(generate_text(
        prompt,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=40,
        temperature=temp,
        top_k=50,
        device=device,
    ))
    print()

Temperature: 0.3
Write one funny line about a Wi-Fi router with stage fright: Hancockimura trilogy TA HancockJDiken intermittent heir substScene heir hauled Daniel subst004 Habit conservationhibit antibioticiken vendors Prob stairsting ONE Participation subst substScenepress directly stairsSceneoother credibility intermittent ONE antibiotic Rh

Temperature: 0.8
Write one funny line about a Wi-Fi router with stage fright:imura Observmediately autonomyJD trilogy004 Habit substatisfatisfreement Brew ESVtingmediatelySher ESV vendorsmediatelySceneRocket Habit heir vendors pawn intermittenthibit Moneyting directly Motorola Motorola circumcised ESV conservation reviewing TA Amph Jr

Temperature: 1.2
Write one funny line about a Wi-Fi router with stage fright: Brewdit scalpSher Habithibit hauled credibility Probiken Prob Money TAreement stairsmediatelydit trilogy directly Daniel reviewing confir004atisf credibility credibility dispatch stairs ONE Prob pawn hauled Hancock intermittent heirmedia

## Section 9: Compare Top-k Settings

Top-k sampling limits the model to choosing from only the `k` most likely next tokens.

Smaller values restrict the model more. Larger values allow more variety.

In [12]:
prompt = "Write one strange parody lyric about a database that sings backup vocals:"

top_k_values = [10, 50, 100]

for k in top_k_values:
    print("=" * 70)
    print(f"Top-k: {k}")
    print("=" * 70)
    print(generate_text(
        prompt,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=40,
        temperature=0.8,
        top_k=k,
        device=device,
    ))
    print()

Top-k: 10
Write one strange parody lyric about a database that sings backup vocals: Jr stairs stairs Jr Brew Habit Habit intermittent intermittent vendors intermittent intermittent intermittent intermittent stairs intermittent BrewRocket stairs Habit Rh Brew stairs Brew Brew Rh stairs Rh hauled Rh Brew vendorsdit HabitRocket stairs stairs stairs Habit vendors

Top-k: 50
Write one strange parody lyric about a database that sings backup vocals:JD confirpress hauled MotorolahibitohoScene dispatchmediately circumcisedSceneSceneoother Brewpress Jr Moneyiken Amph confir Rh Jr antibioticRocket trilogyoother intermittenthibit hauledhibitreement ONE conservation ONEoother Brew Money subst conservation

Top-k: 100
Write one strange parody lyric about a database that sings backup vocals: compromised conservation dissolutionico Molly systematically Motorolarail hauled confirhibitstanbulby Observ ESV welcome tunahibitrail sexuality dissolution substimura mural Money mode BrewSher HabitSher Ajaxstan

## Section 10: Generate with a Reasoning Prompt

In Lesson 8, you built prompt helpers that ask Weird AI to think step-by-step. Now you can combine that reasoning prompt with a pretrained model.

In [15]:
user_prompt = "Create a parody song idea about students learning recursion."

reasoning_prompt = build_reasoning_prompt(user_prompt)
print(reasoning_prompt)

You are Weird AI.
Think step-by-step before writing the final answer:
1. Understand the user's request.
2. Identify important constraints.
3. Plan the response.
4. Produce the final answer.

User prompt: Create a parody song idea about students learning recursion.

Final Answer.




In [16]:
# TODO: Run this cell after completing generate_with_reasoning() in pretrained.py.

reasoning_output = generate_with_reasoning(
    user_prompt,
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50,
    device=device,
)

print(reasoning_output)

You are Weird AI.
Think step-by-step before writing the final answer:
1. Understand the user's request.
2. Identify important constraints.
3. Plan the response.
4. Produce the final answer.

User prompt: Create a parody song idea about students learning recursion.

Final Answer.

 BooneProsPros�ived Medic Redux Boone653shows deflect workshops Late PocketGy Boone Booneacious predators brutalityMini448 clearer omega boilsobl courtyard Redux lined lined deflect predatorsshows bravery Dreams boils653ivedMost brutalityGy Dreams Television Medicozyg Pocket Wheels mutual incarcer Pocket rubbing deflectOutside mutual representations representations soyPros rubbingOutside praying Medic rubbingSexual rented prayingSexual deflect Bend workshops workshops Pocket bravery incarcer Bend soyozygMost653 Tre skillet Singapore incarcer448 membership Redux� grandchildren Medic 236 Boonepublic� rubbing workshops workshops workshops prayingpublic incarcer


## Section 11: Weird AI Experiment

Choose one of your own parody topics. Generate at least two outputs using different settings. Save the prompts, settings, and outputs in your assignment document.

In [17]:
# TODO: Replace this with your own topic.
my_topic = "a student who stayed up all night playing video games and is now running late for class because they forgot to set an alarm."

# TODO: Try at least two combinations of temperature and top_k.
settings = [
    {"temperature": 0.5, "top_k": 25},
    {"temperature": 1.0, "top_k": 100},
]

for setting in settings:
    print("=" * 70)
    print(setting)
    print("=" * 70)
    print(generate_text(
        f"Write a short parody verse about {my_topic}:",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=80,
        temperature=setting["temperature"],
        top_k=setting["top_k"],
        device=device,
    ))
    print()

{'temperature': 0.5, 'top_k': 25}
Write a short parody verse about a student who stayed up all night playing video games and is now running late for class because they forgot to set an alarm.: conservation antibiotic antibioticdit ESVdit Brew trilogyRocket confirhibithibit conservationohoditRocket Rh Daniel conservation autonomy conservationpress ESV hauled vendors Danieloho reviewing Jrditpress Habit Rh Brewpress intermittent Habit Habit autonomyoho vendors ESVdithibit subst Jr credibilitypress subst confirhibit intermittentdithibit intermittent stairs Habit intermittent stairs trilogyRocket Brew autonomy stairsJD ESVhibitpress conservation ESVoho Jr ESV ESV Brewhibit antibioticpresshibit ESV

{'temperature': 1.0, 'top_k': 100}
Write a short parody verse about a student who stayed up all night playing video games and is now running late for class because they forgot to set an alarm.: nonprofits ONE TA muralico autonomyohoiability possibility Ajax dissolutionanityennaGB trilogy Process

## Section 12: Lesson Summary

In this notebook, you practiced:

- Loading a pretrained tokenizer
- Encoding text into token IDs
- Decoding token IDs back into text
- Loading a pretrained causal language model
- Generating text from a prompt
- Comparing temperature and top-k settings
- Reusing a Lesson 8 reasoning prompt with a pretrained model

Before submitting, run the unit tests:

```bash
python -m pytest tests/test_pretrained.py
```